In [ ]:
import numpy as np
import pandas as pd
import pathlib as pl

from pyswmm import Output, NodeSeries

In [ ]:
def node_to_dataframe(data_dict):
    data = np.empty(len(data_dict), dtype=[("index", "datetime64[s]"), ("flow", float)])
    for idx, (key, value) in enumerate(data_dict.items()):
        data["index"][idx] = np.datetime64(key)
        data["flow"][idx] = value 
    df = pd.DataFrame.from_dict(data)
    df.set_index("index", inplace=True)
    return df

In [ ]:
import sys
sys.path.append("../common")
from liss_settings import get_scenario_name, get_results_path

# ---- scenario (must match the run made by step2_run_coupled_models) ----------
domain = "gp"
resolution = "coarse"             # "coarse" | "medium" | "high"
mf_couple_freq_hours = 8.0        # MODFLOW <-> D-Flow FM coupling frequency
n_connections = 244               # SWMM <-> MODFLOW connections ACTUALLY resolved
outfall_node = "O3"               # gp_sewer.inp outfall driving the D-Flow tracer
# -----------------------------------------------------------------------------

scenario = get_scenario_name(domain, resolution, mf_couple_freq_hours, n_connections)
results_ws = get_results_path(domain, resolution, mf_couple_freq_hours, n_connections)

# step2 copies this run's SWMM output into the scenario results directory, so the
# plot always matches the coupled run instead of whatever was last left in
# ../swmm/gp/. The old path ../swmm/greenport/greenport_detailedsewer_v4_nosub175
# no longer exists -- that model was replaced by swmm/gp/gp_sewer.inp, whose
# outfall is "O3" ("171" is a junction there, not an outfall).
swmm_out_path = results_ws / "swmm.out"

print("scenario :", scenario)
print("swmm.out :", swmm_out_path)
assert swmm_out_path.is_file(), (
    f"{swmm_out_path} not found -- run step2_run_coupled_models for this scenario first."
)

In [ ]:
swmm_out = Output(str(swmm_out_path))

In [ ]:
outfall = node_to_dataframe(NodeSeries(swmm_out)[outfall_node].total_inflow)
print(f"{outfall_node}: {len(outfall):,} points, "
      f"{outfall.index[0]} -> {outfall.index[-1]}")
print(f"  total inflow  min={outfall['flow'].min():.4g}  "
      f"max={outfall['flow'].max():.4g}  mean={outfall['flow'].mean():.4g} m3/s")
outfall.head()

In [ ]:
outfall.plot()

In [ ]:
swmm_out.close()